In [2]:
!pip install sodapy




[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from sodapy import Socrata
import pandas as pd

In [ ]:
client = Socrata("www.datos.gov.co", None)

results = client.get(
    "jbjy-vk9h",
    limit=5000
)

df = pd.DataFrame(results)

df.head()

In [12]:
client = Socrata("www.datos.gov.co", None)

results = client.get(
    "jbjy-vk9h",
    limit=5000
)

df = pd.DataFrame(results)

# Seleccionar columnas importantes
df = df[[
    "id_contrato",
    "nombre_entidad",
    "proveedor_adjudicado",
    "valor_del_contrato",
    "descripcion_del_proceso",
    "modalidad_de_contratacion"
]]

#filas 
df = df.dropna()

# Convertir 
df["valor_del_contrato"] = df["valor_del_contrato"].astype(float)

df.head()


,id_contrato,nombre_entidad,proveedor_adjudicado,valor_del_contrato,descripcion_del_proceso,modalidad_de_contratacion
0,CO1.PCCNTR.1000001,PARQUES NACIONALES NATURALES DE COLOMBIA - DIR...,INDUSTRIAS GUERRERO Y COMPAÑIA S.A.S.,17897916.0,Contratar a monto agotable por el sistema de p...,Mínima cuantía
1,CO1.PCCNTR.1000002,ESTABLECIMIENTO CARCELARIO PURIFICACION,OLGA YANET ROJAS BARBOSA,1200000.0,Sin Descripcion,Mínima cuantía
2,CO1.PCCNTR.100001,AGENCIA NACIONAL DE DEFENSA JURÍDICA DEL ESTADO,Jessica Lizeth Martínez Gaviria,28568400.0,Prestar servicios de apoyo técnico y operativo...,Contratación Directa (con ofertas)
3,CO1.PCCNTR.100002,AGENCIA NACIONAL DE DEFENSA JURÍDICA DEL ESTADO,JONATHAN ALBER RONDON BARBOSA,27745000.0,Prestar los servicios de apoyo a la Dirección ...,Contratación Directa (con ofertas)
4,CO1.PCCNTR.100003,AGENCIA NACIONAL DE DEFENSA JURÍDICA DEL ESTADO,Luis Andres Ordoñez Gil,45536400.0,Prestar servicios profesionales para apoyar la...,Contratación Directa (con ofertas)


In [13]:
def calcular_riesgo_basico(row):
    riesgo = 0
    alertas = []

    if row["valor_del_contrato"] > 50000000:
        riesgo += 30
        alertas.append("Contrato de alto valor")

    if "directa" in row["modalidad_de_contratacion"].lower():
        riesgo += 30
        alertas.append("Contratación directa")

    if len(row["descripcion_del_proceso"]) < 50:
        riesgo += 20
        alertas.append("Descripción insuficiente")

    return riesgo, alertas

In [14]:
df["riesgo_base"], df["alertas_base"] = zip(*df.apply(calcular_riesgo_basico, axis=1))

df.head()

,id_contrato,nombre_entidad,proveedor_adjudicado,valor_del_contrato,descripcion_del_proceso,modalidad_de_contratacion,riesgo_base,alertas_base
0,CO1.PCCNTR.1000001,PARQUES NACIONALES NATURALES DE COLOMBIA - DIR...,INDUSTRIAS GUERRERO Y COMPAÑIA S.A.S.,17897916.0,Contratar a monto agotable por el sistema de p...,Mínima cuantía,0,[]
1,CO1.PCCNTR.1000002,ESTABLECIMIENTO CARCELARIO PURIFICACION,OLGA YANET ROJAS BARBOSA,1200000.0,Sin Descripcion,Mínima cuantía,20,[Descripción insuficiente]
2,CO1.PCCNTR.100001,AGENCIA NACIONAL DE DEFENSA JURÍDICA DEL ESTADO,Jessica Lizeth Martínez Gaviria,28568400.0,Prestar servicios de apoyo técnico y operativo...,Contratación Directa (con ofertas),30,[Contratación directa]
3,CO1.PCCNTR.100002,AGENCIA NACIONAL DE DEFENSA JURÍDICA DEL ESTADO,JONATHAN ALBER RONDON BARBOSA,27745000.0,Prestar los servicios de apoyo a la Dirección ...,Contratación Directa (con ofertas),30,[Contratación directa]
4,CO1.PCCNTR.100003,AGENCIA NACIONAL DE DEFENSA JURÍDICA DEL ESTADO,Luis Andres Ordoñez Gil,45536400.0,Prestar servicios profesionales para apoyar la...,Contratación Directa (con ofertas),30,[Contratación directa]


In [ ]:
""" from sodapy import Socrata
import pandas as pd

client = Socrata("www.datos.gov.co", None)

chunk_size = 50000
offset = 0
all_data = []

while True:
    print(f"Descargando desde {offset}...")
    
    results = client.get(
        "jbjy-vk9h",
        limit=chunk_size,
        offset=offset
    )
    
    if not results:
        break
    
    all_data.extend(results)
    offset += chunk_size

df = pd.DataFrame(all_data)

print(df.shape) """